In [9]:
import sys
import os

# Use absolute paths to ensure imports work from anywhere
pipeline_dir = r"C:\Users\alexnewnham\Dropbox\MobRes\Scripts\CASD\Sandbox\Masslayoff\pipeline"
eda_dir = r"C:\Users\alexnewnham\Dropbox\MobRes\Scripts\CASD\Sandbox\Masslayoff\pipeline\eda"

sys.path.insert(0, eda_dir)
sys.path.insert(0, pipeline_dir)

import polars as pl
import numpy as np
import pandas as pd
from config import *
from eda_base import *
from eda_stats import *

# Establishment Data (S0 Output)
Overview of parsed establishments from DADS

In [10]:
# Load sample year to understand structure
etab_2010 = pl.scan_parquet(os.path.join(OUT_DIR, 'etab', 'etab_2010.parquet')).collect()
print(f"Shape: {etab_2010.shape}")
print(f"\nColumns: {etab_2010.columns}")


Shape: (71444, 16)

Columns: ['siret', 'an', 'eff_3112_et', 'eff_decl_et', 'eff_moy_et', 'eff_0101_et', 'nb_postes_n', 'nb_sal', 'siren', 'nic', 'comt', 'dept', 'apet', 'apet_2d', 'comt_raw', 'is_ghost']


In [11]:
# Time dimension: counts across all years
year_counts = []
for year in PROJECT_YEARS:
    try:
        path = os.path.join(OUT_DIR, 'etab', f'etab_{year}.parquet')
        n = pl.scan_parquet(path).select(pl.len()).collect()["len"][0]
        year_counts.append({"year": year, "n_establishments": n})
    except:
        pass
        
df_counts = pl.DataFrame(year_counts)
print("Establishments by year:")
print(df_counts)


Establishments by year:
shape: (14, 2)
┌──────┬──────────────────┐
│ year ┆ n_establishments │
│ ---  ┆ ---              │
│ i64  ┆ i64              │
╞══════╪══════════════════╡
│ 2002 ┆ 65199            │
│ 2003 ┆ 64988            │
│ 2004 ┆ 65011            │
│ 2005 ┆ 65085            │
│ 2006 ┆ 65152            │
│ …    ┆ …                │
│ 2011 ┆ 71551            │
│ 2012 ┆ 71762            │
│ 2013 ┆ 71931            │
│ 2014 ┆ 72063            │
│ 2015 ┆ 72249            │
└──────┴──────────────────┘


In [12]:
# Missing values across key columns
missing_cols = ['siren', 'siret', 'comt', 'ze2010_code', 'apet_2d']
missing_stats = etab_2010.select([
    pl.col(c).is_null().sum().alias(f"{c}_null") for c in missing_cols if c in etab_2010.columns
])

print("Missing values (2010):")
for c in missing_cols:
    if c in etab_2010.columns:
        null_count = missing_stats[f"{c}_null"][0]
        pct = 100 * null_count / len(etab_2010)
        print(f"  {c}: {null_count} ({pct:.1f}%)")


Missing values (2010):
  siren: 0 (0.0%)
  siret: 0 (0.0%)
  comt: 4652 (6.5%)
  apet_2d: 4281 (6.0%)


In [13]:
# Spatial dimension: communes and zones
spatial_summary = etab_2010.group_by('comt').agg(pl.len().alias('n_estab')).sort('n_estab', descending=True).head(10)
print("Top 10 communes by establishment count (2010):", spatial_summary)


Top 10 communes by establishment count (2010): shape: (10, 2)
┌───────┬─────────┐
│ comt  ┆ n_estab │
│ ---   ┆ ---     │
│ str   ┆ u32     │
╞═══════╪═════════╡
│ 75056 ┆ 5271    │
│ null  ┆ 4652    │
│ 13055 ┆ 941     │
│ 31555 ┆ 651     │
│ 69123 ┆ 646     │
│ 06088 ┆ 496     │
│ 44109 ┆ 455     │
│ 34172 ┆ 444     │
│ 67482 ┆ 443     │
│ 33063 ┆ 381     │
└───────┴─────────┘


In [14]:
# Sector distribution
sector_summary = etab_2010.group_by('apet_2d').agg(pl.len().alias('n_estab')).sort('n_estab', descending=True).head(15)
print("Top 15 sectors (2010):", sector_summary)


Top 15 sectors (2010): shape: (15, 2)
┌─────────┬─────────┐
│ apet_2d ┆ n_estab │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ 84      ┆ 10919   │
│ 47      ┆ 8788    │
│ 56      ┆ 4314    │
│ null    ┆ 4281    │
│ 86      ┆ 3356    │
│ …       ┆ …       │
│ 41      ┆ 2288    │
│ 10      ┆ 2182    │
│ 88      ┆ 2029    │
│ 71      ┆ 1958    │
│ 43      ┆ 1958    │
└─────────┴─────────┘


In [15]:
# Consistency check: SIREN-SIRET relationship
# Each SIRET should be unique, each SIREN can have multiple SIRETs
siren_siret = etab_2010.group_by('siren').agg(pl.col('siret').n_unique().alias('n_sirets')).sort('n_sirets', descending=True).head(10)
print("Firms (SIREN) by number of establishments (SIRET) - top 10 (2010):", siren_siret)


Firms (SIREN) by number of establishments (SIRET) - top 10 (2010): shape: (10, 2)
┌───────────┬──────────┐
│ siren     ┆ n_sirets │
│ ---       ┆ ---      │
│ str       ┆ u32      │
╞═══════════╪══════════╡
│ 156403354 ┆ 421      │
│ 126915214 ┆ 413      │
│ 105877062 ┆ 409      │
│ 173475211 ┆ 408      │
│ 115688335 ┆ 402      │
│ 174499822 ┆ 394      │
│ 125233189 ┆ 390      │
│ 135913093 ┆ 385      │
│ 116617036 ┆ 385      │
│ 110082278 ┆ 385      │
└───────────┴──────────┘
